If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec21_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 21: Causality — the Randomized Controlled Experiment
v.ekc

The botox experiment, tested properly — and why random assignment lets us say *"the treatment works"* instead of just *"the groups differ."* (Slides: KL Lecture 21 deck.)

**Today's sections:**
1. The botox RCT
2. Testing the hypothesis
3. Discussion — Super Soda

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. The Botox RCT 💉

31 back-pain patients, randomly assigned. Last time you found these counts and proportions yourself — here they are as worked cells.

In [ ]:
botox = Table.read_table('bta.csv')
botox.show()

In [ ]:
# How many people experienced pain relief in each group?
botox.group('Group',sum)

In [ ]:
# What proportion of people experienced pain relief in each group?
botox.group('Group',np.average)

---
## 2. Testing the Hypothesis

> **Null:** botox does nothing — the observed difference is just the luck of random assignment.
> **Alternative:** botox increases the chance of pain relief.
> **Test statistic:** difference in proportion relieved (treatment − control); *big* values favor the alternative.

(Writing tips, KL deck slides 15–16: make the null precise enough to simulate, and say which direction of the statistic favors the alternative.)

In [ ]:
def difference_of_means(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
def one_simulated_difference(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, label, 'Shuffled Label')   

In [ ]:
observed_diff = difference_of_means(botox, 'Result', 'Group')
observed_diff

In [ ]:
one_simulated_difference(botox, 'Result', 'Group')

In [ ]:
simulated_diffs = make_array()

for i in np.arange(10000):
    sim_diff = one_simulated_difference(botox, 'Result', 'Group')
    simulated_diffs = np.append(simulated_diffs, sim_diff)

In [ ]:
col_name = 'Distances between groups'
Table().with_column(col_name, simulated_diffs).hist(col_name)
plots.scatter(observed_diff,-0.01, c='red')

In [ ]:
# p-value
sum(simulated_diffs >= observed_diff)/len(simulated_diffs)

**Conclusion**: The test favors the alternative hypothesis over the null. The result is statistically significant. Therefore, the data supports the hypothesis that the treatment is doing something. Because the trials were randomized, the test is evidence that the treatment *causes* the difference. The random assignment of patients to the two groups ensures that there is no confounding variable that could affect the conclusion of causality.

> **And because assignment was random**, "not chance" leaves only the treatment: this is evidence of *causation*, not just association. This is the same conclusion structure Homework 7 asks you to write out.

---
## 3. Discussion: Super Soda 🥤

In a taste test, 91 of 200 tasters prefer Super Soda over its rival. The company claims a 50/50 split. Before scrolling: what are the null, alternative, and test statistic?

### Try It 1: Set up the test 🏷️

1. Write the null hypothesis, the alternative, and the test statistic in the cell below — then check against the simulation.

*1. Your answer: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Null: tasters are 50/50, like fair-coin tosses — 91 is chance. Alternative: fewer than half prefer Super Soda. Test statistic: number of "heads" (Super Soda preferences) in 200; small values favor the alternative.*

</details>

In [ ]:
def simulate_one_count(sample_size):
    return np.count_nonzero(np.random.choice(['H', 'T'], sample_size) == 'H')
simulate_one_count(200)

In [ ]:
num_simulations = 10000
counts = make_array()
for i in np.arange(num_simulations):
    counts = np.append(counts, simulate_one_count(200))

In [ ]:
trials = Table().with_column('Number of Heads', counts)
trials.hist(right_end=91)
plots.ylim(-0.001, 0.055)
plots.scatter(91, 0, color='red', s=40, zorder=3)
plots.title('Prediction Under the Null');

In [ ]:
np.count_nonzero(counts <= 91)/len(counts)

> **Verdict:** p ≈ 0.11 — more than one in ten fair-split worlds produce a result this low. By the 5% convention, we *can't* reject the null. Compare with botox: same recipe, different conclusion, and both are useful answers.

---
> **Today's takeaway:** random assignment + hypothesis test = causal evidence; and a big p-value is a finding too, not a failure.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — RCT test
```python
observed = difference_of_means(t, value_label, group_label)
# shuffle labels 10,000 times, then:
p_value = sum(simulated >= observed) / len(simulated)
```